# Notebook 21 — Scientific Paper RAG (PDF Ingestion)
Goal: Build a simple RAG system that ingests scientific PDFs and answers questions using retrieved text.

This notebook demonstrates the realistic workflow:

PDF → text extraction → chunking → embeddings → retrieval → prompt → answer

## Section 1 — Install and Import Libraries

In [ ]:
# If needed:
# pip install pymupdf sentence-transformers scikit-learn transformers torch pandas numpy

import fitz  # pymupdf
import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import pipeline

## Section 2 — Load a PDF
Replace the filename with one of your own scientific papers.

In [ ]:
pdf_path = "example_paper.pdf"

doc = fitz.open(pdf_path)

pages = []
for page in doc:
    pages.append(page.get_text())

full_text = "\n".join(pages)

len(full_text)

## Section 3 — Inspect Extracted Text

In [ ]:
print(full_text[:2000])

## Section 4 — Chunk the Paper
Split the paper into smaller chunks for embedding.

In [ ]:
def chunk_text(text, max_sentences=3):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences]).strip()
        if chunk:
            chunks.append(chunk)
            
    return chunks

chunks = chunk_text(full_text)

len(chunks)

In [ ]:
chunks[:5]

## Section 5 — Create Embeddings for Chunks

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(chunks)

len(chunk_embeddings), chunk_embeddings[0].shape

## Section 6 — Retrieval Function

In [ ]:
def retrieve(query, top_k=4):
    
    query_emb = embed_model.encode([query])
    
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    
    idx = np.argsort(sims)[::-1][:top_k]
    
    return [chunks[i] for i in idx]

query = "What experimental model was used in this study?"

retrieve(query)

## Section 7 — Build a Scientific RAG Prompt

In [ ]:
def build_prompt(query, context_chunks):
    
    context = "\n".join(context_chunks)
    
    prompt = f"""You are a scientific research assistant.

Answer the question using ONLY the provided paper excerpts.
If the answer cannot be found in the excerpts, say so.

Paper Excerpts:
{context}

Question:
{query}

Answer:"""
    
    return prompt

context = retrieve(query)

prompt = build_prompt(query, context)

print(prompt[:2000])

## Section 8 — Generate an Answer

In [ ]:
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=120
)

result = generator(prompt)[0]["generated_text"]

print(result)

## Section 9 — Inspect Retrieved Evidence

In [ ]:
for c in context:
    print("-" * 60)
    print(c)

## Section 10 — Example Scientific Queries
Try questions such as:
- What datasets were used?
- What statistical model was applied?
- What biological mechanism is proposed?
- What experimental system was used?

## Section 11 — Evaluation Checklist
When evaluating a RAG answer:

1. Did retrieval return the correct chunk?
2. Does the answer appear in the retrieved context?
3. Did the model hallucinate anything?
4. Would different chunk sizes improve retrieval?
5. Would top_k retrieval help?

## Skills Gained
- ingesting PDFs
- extracting scientific text
- building a RAG index
- querying scientific papers
- evaluating RAG answers

This notebook demonstrates a realistic research RAG workflow.